# Lecture - Pandas basics

- pandas.Series
- pandas.DataFrame
- read_csv
- indexing
- plotting

## Pandas Series

- can create from dictionary
- can create from list
- can create from np.array

In [ ]:
import pandas as pd 

programs_dict = dict(AI = 26, NET = 38, Java = 30, UX = 28)

programs_series = pd.Series(programs_dict)

programs_series

In [ ]:

# extract values through indexing 
print(f"{programs_series[0] = }")
print(f"{programs_series[-1] = }")

print(f"{programs_series['UX'] = }")

# get keys 
print(f"{programs_series.keys() = }")
print(f"{programs_series.keys()[0] = }")

In [ ]:
import random as rnd 

# for reproducibility - gets same set of values
rnd.seed(1337)

dice_series = pd.Series([rnd.randint(1,6) for _ in range(10)])
dice_series.head() # inspects the five first rows

In [ ]:
dice_series

In [ ]:
print(f"{dice_series.min() = }")
print(f"{dice_series.argmin() = }") # gives index for min value
print(f"{dice_series.max() = }") 
print(f"{dice_series.mean() = }") # average  
print(f"{dice_series.median() = }") # sort all values in order - pick the middle one, if middle are 2 numbers calculate average of them

---
## DataFrame

- tabular data with rows and columns 
- analog to 2D numpy arrays with flexible row indices and col names
- "specialized" dictionary with col name mapped to a Series object

In [ ]:
# instantiated a DataFrame from a Series object
df_programs = pd.DataFrame(programs_series, columns=("Number_of_students",))
df_programs

In [ ]:
# create 2 Series objects 
students = pd.Series({"AI": 26, "NET": 38, "UX": 28, "Java": 30})
skills = pd.Series({"AI": "Python", "net": "C#", "UX": "Figma", "Java": "Java"})

# create a DataFrame from 2 Series objects
df_programs = pd.DataFrame({"Students": students, "Skills": skills })
df_programs

In [ ]:
df_programs["Students"]

In [ ]:
df_programs["Students"].mean(), (26+30+38+28)/4

In [ ]:
median_student_number = df_programs["Students"].median()
print(f"Median students in the programs {df_programs.index.to_list()} is {median_student_number:.0f}")

In [ ]:
# create 2 Series objects 
students = pd.Series({"AI": 26, "NET": 38, "UX": 28, "Java": 30})
skills = pd.Series({"AI": "Python", "NET": "C#", "UX": "Figma", "Java": "Java"})

# create a DataFrame from 2 Series objects
df_programs = pd.DataFrame({"Students": students, "Skills": skills })
df_programs

In [ ]:
df_programs["Skills"]

In [ ]:
df_programs["Skills"][0], df_programs["Skills"]["AI"], df_programs["Skills"]["UX"],  


## Indexers 

- loc - slicing and indexing using explicit index
- iloc - slicing and indexing using Python-style index

In [ ]:
df_programs

In [ ]:
df_programs.loc["AI"] # returns a Series object

In [ ]:
df_programs.loc["Java"]

In [ ]:
df_programs.loc[["AI", "UX"]]

In [ ]:
df_programs.iloc[1:3] # returns a DataFrame object

## Masking

In [ ]:
df_programs

In [ ]:
df_programs["Students"] >= 30

In [ ]:
# using masking to filter the DataFrame
df_programs_over_29 = df_programs[df_programs["Students"] >= 30]
df_programs_over_29

In [ ]:
df_programs

---
## Excel data

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

df = pd.read_excel("../Data/calories.xlsx")
df.head()

In [ ]:
df.info()

In [ ]:
df.shape # (rows, columns)

In [ ]:
# liquid food and solid food
df["FoodCategory"].unique()

In [ ]:
df["per100grams"].unique()

## Data cleaning and explorations

- type convert string objects with numerical values to int
- change column names
- separate into liquids and solids



In [ ]:
df.head()

In [ ]:
# renaming columns
df = df.rename(
    dict(Cals_per100grams="Calories", KJ_per100grams="kJ", per100grams="per100"),
    axis="columns",
)

df.head()


In [ ]:
# converting Calories to int
df["Calories"] = df["Calories"].str[:-3].astype(int)
df.head()

In [ ]:
df["Calories"].head()

In [ ]:
# counts the frequency of each unique element
df["per100"].value_counts()

In [ ]:
(df["per100"] == "100g").value_counts()

In [ ]:
# separate liquids and solids
# filter out all solids
df_solids = df[df["per100"] == "100g"]

# filter out all liquids
df_liquids = df[df["per100"] == "100ml"]

df_liquids.head()

In [ ]:
df_solids.head()

In [ ]:
# check that we only have 100g in per100
df_solids["per100"].value_counts()

## Assignment from stakeholder (body builder)

- find the most calory rich solid
- find the most calory rich liquids

In [ ]:
df_solids.sort_values(by="Calories")

In [ ]:
solids_sorted_calories = df_solids.sort_values(by="Calories", ascending = False) # ascending - False -> descending
solids_sorted_calories.iloc[:10]

In [ ]:
solids_top5_calories = solids_sorted_calories.head()
solids_top5_calories

In [ ]:
liquids_top5_calories = df_liquids.sort_values(by="Calories", ascending = False).head()
liquids_top5_calories

In [ ]:
df["FoodCategory"].value_counts()

In [ ]:
# top five food categories by calories
top5_category_median = df.groupby("FoodCategory").median("Calories").sort_values(by = "Calories", ascending=False).head().reset_index()
top5_category_median

In [ ]:
solids_top5_calories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=120)
titles = ["Solid top 5 calories", "Liquids top 5 calories", "Top 5 calories per group (median)"]
data_frames = (solids_top5_calories, liquids_top5_calories, top5_category_median)
x_columns = ["FoodItem", "FoodItem", "FoodCategory"]


for i, (data, title) in enumerate(zip(data_frames, titles)):
    sns.barplot(data=data, x = x_columns[i], y = "Calories", ax = axes[i])
    axes[i].set(title = title)
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation = 90)

plt.savefig("../assets/top_calories_food.png", facecolor = "white", bbox_inches = "tight")